# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-cou

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
htt

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [11]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'professional profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co'},
  {'type': 'brand/about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Zhihu page', 'url': 'https://www.zhihu.com/org/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-4.7
Updated
5 days ago
•
28k
•
1.09k
Qwen/Qwen-Image-Layered
Updated
9 days ago
•
15.3k
•
802
MiniMaxAI/MiniMax-M2.1
Updated
about 16 hours ago
•
45.3k
•
471
Qwen/Qwen-Image-Edit-2511
Updated
5 days ago
•
16.6k
•
467
google/functiongemma-270m-it
Updated
10 days ago
•
35.4k
•
658
Browse 2M+ models
Spaces
Running
on
Zero
Featured
596
TRELLIS.2
🏢
596
High-fidelity 3D Generation from images
Running
Featured
3.03k
Wan2.2 Animate
👁
3.03k
Wan2.2 Animate
Running
on
Zero
Featured
298
Qwen Image Layered
🚀
298
Decompose an image int

In [25]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

#brochure_system_prompt = """
#You are an assistant that analyzes the contents of several relevant pages from a company website
#and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
#Respond in markdown without code blocks.
#Include details of company culture, customers and careers/jobs if you have the information.
#"""


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-4.7\nUpdated\n5 days ago\n•\n28k\n•\n1.09k\nQwen/Qwen-Image-Layered\nUpdated\n9 days ago\n•\n15.3k\n•\n802\nMiniMaxAI/MiniMax-M2.1\nUpdated\nabout 16 hours ago\n•\n45.3k\n•\n471\nQwen/Qwen-Image-Edit-2511\nUpdated\n5 days ago\n•\n16.6k\n•\n467\ngoogle/functiongemma-270m-it\nUpdated\n10 days ago\n•\n35.4k\n•\n658\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n596\nTRELL

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


# Welcome to Hugging Face: The Friendly Face of AI 🤗

---

## Who Are We?

At Hugging Face, we're more than just a company—we're an *AI community* on a mission to build the **future** of machine learning. Think of us as the bustling town square for ML enthusiasts, where the smartest minds come together to collaborate, create, and caffeinate (virtually, of course).

We provide the ultimate playground for over **2 million machine learning models**, **half a million datasets**, and a million+ cutting-edge applications. Whether you want to generate images from text prompts, dive into high-fidelity 3D generation, or tinker with face swaps and image edits, Hugging Face is your one-stop AI shop.

---

## What Makes Us Hugging Face?

- **Community-driven:** No lone wolves here. We thrive on open source and collaboration. Whether you're a newbie, a world-class scientist, or somewhere in-between, there’s a comfy spot for you at the table.

- **Diverse AI modalities:** Text, images, video, audio, even *3D*—you name it, we host it.

- **Fast and Furious ML:** Powered by our open-source stack, we help you move faster than a caffeinated algorithm. Build your portfolio, share your work, and explore innovations worldwide.

- **Enterprise-grade, but with heart:** Got a team that needs serious security, access control, and dedicated support? Our paid Compute and Enterprise solutions have you covered without losing our signature friendliness.

---

## Who Uses Hugging Face?

- **Machine Learning Engineers & Scientists:** To share models, datasets, and collaborate on tomorrow’s breakthroughs.

- **Innovators & Developers:** To build AI apps that wow users.

- **Enterprises:** To deploy and scale AI with rock-solid security.

- **You:** Seriously, if you love ML, there's a place for you here. And if you don’t yet, we promise you’ll love it once you join!

---

## Dive Into Our Playground

- **Models:** Browse and download from over **2 million models**—including fan favorites like zai-org/GLM-4.7 and MiniMaxAI/MiniMax-M2.1.

- **Spaces:** Create and explore AI apps running right on the platform—from amazing 3D generators to slick image editors.

- **Datasets:** Access over **500,000 datasets** to train your next great AI.

- **Community:** Join discussions, share feedback, and collaborate with AI enthusiasts across the globe.

---

## Ready to Join Our Team?

We’re shaping the future of AI *with* and *for* humans. If you want to combine your passion for machine learning with a culture that values openness, creativity, and fun, Hugging Face might just be your next home.

Work with us, and you’ll be...

- Part of a rapidly growing, diverse, and passionate team.

- Empowered to build, share, and scale open and ethical AI.

- Supported by a community that values knowledge, fun, and a little bit of whimsy.

---

## Why Hugging Face? Because AI Should Be Friendly 😄

From the smiles in our logos to the spirit of collaboration that fuels our community, Hugging Face believes AI is best when it’s open, accessible, and built by many hands (and friendly faces).

---

### Get Cozy With Us

- **Sign Up:** Build your ML portfolio and join our vibrant community.

- **Explore:** Browse models, datasets, and AI apps that amaze.

- **Collaborate:** Whether you’re a coder, researcher, or dreamer, find your people here.

- **Accelerate:** Use our enterprise-grade tools to bring your AI projects to life faster.

---

*Join Hugging Face — where machine learning gets a (literal) hug and a high five.* 🤗✋

Visit us at [huggingface.co](https://huggingface.co)

---

### Fun Fact

Our brand colors are #FFD21E (a sunny yellow), #FF9D00 (a friendly orange), and #6B7280 (a reliable gray)—because AI should be bright, warm, and dependable. Just like us!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# Hugging Face: The AI Community Hugging You Into the Future 🤗

---

## Who Are We?

Welcome to **Hugging Face**, the world's coziest corner of the internet where machine learning (ML) minds unite to build the future 🤖✨. We're the machine learning community’s favorite playground, buzzing with over 2 million models, hundreds of thousands of datasets, and an ever-growing troop of brilliant brains. Whether you speak fluent Python, dream in tensors, or just love the magic of AI, we have a spot just for you!

---

## What’s the Buzz About?

We’re *more than* just a website. Hugging Face is:

- **The Hub**: A central, open-source wonderland where anyone—from curious beginners to AI wizards—can share, explore, and collaborate on ML models, datasets, and applications.
- **Multi-Modal Magic**: From text, images, video, audio to 3D, we support it all. Bring your wildest AI dreams; we’ve got the tools.
- **Spaces**: Run your cool AI apps and demos live on the cloud for the world to see (and marvel at).
- **Compute Power**: Ready for takeoff with paid compute and enterprise-grade solutions, so your AI can *actually* crunch numbers fast.

Fun fact: People are already making AI that edits images, animates, generates 3D art, and even layers images like a pro PowerPoint presenter—all right here!

---

## Culture & Community ✨

- **Open & Ethical AI**: We believe great AI should be open, inclusive, and ethical. No dark sorcery here—just bright minds transforming the world.
- **Collaborative Spirit**: If AI is a party, we’re the hosts making sure everyone—from newbie to pro—gets to dance.
- **Learning & Sharing**: Build your AI portfolio, learn from others, and become part of a fast-growing community that’s changing tech.

We put our heart (and face!) into supporting the next generation of machine learning scientists, engineers, and dare-we-say-it, AI superheroes.

---

## For Our Prospective Customers & Enterprise Users

**Give your team the superpowers they deserve** with Hugging Face Enterprise:

- Enterprise-grade security (because your AI secrets deserve Fort Knox).
- Single Sign-On and granular access controls to keep your data safe and sound.
- Analytics dashboards to track your ML empire.
- Private datasets, private storage, and private everything to collaborate in stealth mode.
- Advanced compute options for when "fast" just won't cut it.

Starting at $20/user/month. Because building the future shouldn't break the bank.

---

## Careers: Join the Hug!

Are you passionate about AI, open source, and community? Ready to work where your coffee breaks might include brainstorming the next big breakthrough in ML with brilliant, quirky folks who share your love for all things nerdy?

Check out our [Careers Page](https://huggingface.co/careers) — Whether you’re a developer, ML scientist, or community builder, we want you on the team! Perks include working on edge-of-tech projects, a vibrant international community, and yes, lots of virtual hugs 🤗.

---

## Why Hugging Face?

- **2M+ Models & 500K+ Datasets**—More options than you can shake a neural net at.
- **Open Source First**—Freedom to innovate without the corporate chains.
- **Community-Powered**—Thousands of contributors, millions of users, one shared mission.
- **Tools for Everyone**—From hobbyists to enterprises, AI is for all.

---

## Ready to Hug the Future?

- Explore AI apps and models.
- Share your own creations.
- Collaborate, learn, and grow.

Jump in and **sign up today**! Because the future of AI isn’t just built with code — it’s built with a community that cares. 🤗

---

**Hugging Face**  
*Making AI less scary, more friendly.*  

Colors to brighten your ML day:  
🟨 #FFD21E | 🟧 #FF9D00 | ⚙️ #6B7280  

Follow the hugging revolution:  
[GitHub](https://github.com/huggingface) | [Twitter](https://twitter.com/huggingface) | [LinkedIn](https://www.linkedin.com/company/huggingface) | [Discord](https://discord.gg/huggingface)

---

*Hugging Face: Where machine learning meets a warm smile.*

In [26]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 18 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the vibrant AI community and collaboration platform building the future of machine learning. With a mission to empower the next generation of machine learning engineers, scientists, and end users, Hugging Face serves as a central hub where the global ML community shares, explores, discovers, and experiments with open-source machine learning models, datasets, and applications.

---

## What We Offer

- **Model Hub:** Access over 2 million pre-trained machine learning models across modalities including text, image, video, audio, and 3D. Collaborate and contribute to widely used models such as GLM-4.7 and Qwen Image Layered.
  
- **Datasets:** Explore a vast repository featuring over 500,000 datasets covering diverse domains enabling faster experimentation and innovation.

- **Spaces:** Host and discover AI applications and interactive demos (1M+ applications) running on our infrastructure, facilitating easy sharing and usage of ML-powered tools.

- **Open Source Stack:** Accelerate your development with Hugging Face’s open-source libraries designed for rapid iteration and experimentation.

- **Enterprise Solutions:** For teams and businesses, Hugging Face provides advanced, secure platforms with enterprise-grade security, access controls, and dedicated support for building AI solutions at scale.

---

## Community & Collaboration

Hugging Face isn’t just a platform; it’s a thriving community of passionate machine learning professionals and enthusiasts who contribute openly to a shared goal of an ethical and innovative AI future. Users worldwide build portfolios, collaborate on projects, and leverage community-driven resources to push the limits of what's possible in AI.

---

## Company Culture

- **Open and Inclusive**: Hugging Face fosters an open-source ethos, encouraging contributions, knowledge-sharing, and ethical development practices.
  
- **Innovation-Driven:** With cutting-edge AI technologies and a fast-growing community, the company promotes innovative thinking and collaboration.
  
- **User-Centric:** Focused on empowering users at all levels, from beginners to experts, offering tools and support designed to facilitate learning and growth.

---

## Customers & Users

Hugging Face serves:

- Individual ML researchers and enthusiasts seeking to access and share models and datasets.
- Teams and enterprises looking for scalable, secure platforms to develop and deploy AI solutions.
- Developers of AI applications across industries using Hugging Face Spaces for deployment.
- Educational institutions and students building their knowledge and profiles in machine learning.

---

## Careers & Join Us

Join Hugging Face to be part of a forward-thinking AI community dedicated to transparency, ethics, and democratization of machine learning. The company offers:

- Opportunities to work with state-of-the-art machine learning models and infrastructure.
- A collaborative environment with a global and diverse team.
- Roles for engineers, researchers, community managers, and enterprise specialists passionate about AI.

Explore open positions and build your career advancing the future of AI at Hugging Face.

---

## Connect & Explore

- Visit [huggingface.co](https://huggingface.co) to browse models, datasets, and AI applications.
- Join the community to start sharing, learning, and growing.
- Discover enterprise and team solutions to accelerate your ML projects securely.
- Sign up to build your portfolio, collaborate, and accelerate AI innovation.

---

**Hugging Face — The AI community building the future, together.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>